In [1]:
import pandas as pd
import requests
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor

pio.renderers.default = "notebook_connected"

session = requests.Session()

In [2]:
def load_weekly_peak_data(dataset_id, period_label, peak_label, start_hour, end_hour):
    url = f"https://data.ny.gov/resource/{dataset_id}.json"

    params = {
        "$select": "date_extract_y(transit_timestamp) as year,"
                   "date_extract_woy(transit_timestamp) as week_of_year,"
                   "borough,"
                   "avg(ridership) as avg_ridership,"
                   "sum(ridership) as total_ridership",
        "$where": f"transit_mode = 'subway' "
                  f"AND borough IN('Manhattan','Brooklyn','Queens','Bronx') "
                  f"AND date_extract_hh(transit_timestamp) >= {start_hour} "
                  f"AND date_extract_hh(transit_timestamp) <= {end_hour}",
        "$group": "year,week_of_year,borough",
        "$limit": 500000
    }

    r = session.get(url, params=params, timeout=120)
    r.raise_for_status()
    data = r.json()

    if not data:
        raise ValueError(f"No data returned for {dataset_id} - {peak_label}")

    df = pd.DataFrame(data)

    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["week_of_year"] = pd.to_numeric(df["week_of_year"], errors="coerce")
    df["avg_ridership"] = pd.to_numeric(df["avg_ridership"], errors="coerce")
    df["total_ridership"] = pd.to_numeric(df["total_ridership"], errors="coerce")

    df = df.dropna(subset=["year", "week_of_year", "borough", "avg_ridership", "total_ridership"]).copy()

    df["year"] = df["year"].astype(int)
    df["week_of_year"] = df["week_of_year"].astype(int)
    df["peak_period"] = peak_label
    df["period"] = period_label

    df["week_start"] = pd.to_datetime(
        df["year"].astype(str) + "-W" + df["week_of_year"].astype(str).str.zfill(2) + "-1",
        format="%G-W%V-%u",
        errors="coerce"
    )

    df = df.dropna(subset=["week_start"]).copy()

    return df

In [4]:
def load_all_data_parallel():
    jobs = [
        ("wujg-7c2s", "Pre Policy", "Morning Peak", 7, 10),
        ("wujg-7c2s", "Pre Policy", "Evening Peak", 16, 19),
        ("5wq4-mkjj", "Post Policy", "Morning Peak", 7, 10),
        ("5wq4-mkjj", "Post Policy", "Evening Peak", 16, 19),
    ]

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = [
            executor.submit(load_weekly_peak_data, dataset_id, period_label, peak_label, start_hour, end_hour)
            for dataset_id, period_label, peak_label, start_hour, end_hour in jobs
        ]
        results = [f.result() for f in futures]

    return pd.concat(results, ignore_index=True)

In [5]:
df = load_all_data_parallel()

print("Rows loaded after full API aggregation:", len(df))
print(df.head())

Rows loaded after full API aggregation: 2592
   year  week_of_year    borough  avg_ridership  total_ridership  \
0  2020             1      Bronx      44.369048         409970.0   
1  2020             1   Brooklyn      54.234139        1099326.0   
2  2020             1  Manhattan      91.483153        1487882.0   
3  2020             1     Queens      78.257863         763875.0   
4  2020             2      Bronx      67.879076         907679.0   

    peak_period      period week_start  
0  Morning Peak  Pre Policy 2019-12-30  
1  Morning Peak  Pre Policy 2019-12-30  
2  Morning Peak  Pre Policy 2019-12-30  
3  Morning Peak  Pre Policy 2019-12-30  
4  Morning Peak  Pre Policy 2020-01-06  


In [6]:
peak_selector = widgets.Dropdown(
    options=["Morning Peak", "Evening Peak"],
    value="Morning Peak",
    description="Peak Hour:"
)

borough_selector = widgets.Dropdown(
    options=["All", "Manhattan", "Brooklyn", "Queens", "Bronx"],
    value="All",
    description="Borough:"
)

output = widgets.Output()

In [7]:
def render_dashboard(*args):
    with output:
        output.clear_output(wait=True)

        data = df.copy()
        data = data[data["peak_period"] == peak_selector.value]

        if borough_selector.value != "All":
            data = data[data["borough"] == borough_selector.value]

        weekly = (
            data.groupby(["week_start", "period"], as_index=False)["avg_ridership"]
            .mean()
            .sort_values(["week_start", "period"])
        )

        fig1 = px.line(
            weekly,
            x="week_start",
            y="avg_ridership",
            color="period",
            title="Weekly Subway Ridership Trend"
        )
        fig1.update_layout(
            width=950,
            height=430,
            xaxis_title="Week",
            yaxis_title="Average Ridership",
            legend_title=""
        )
        fig1.update_xaxes(nticks=12, tickformat="%Y-%m")
        display(fig1)

        borough_avg = (
            data.groupby(["borough", "period"], as_index=False)["avg_ridership"]
            .mean()
            .sort_values(["borough", "period"])
        )

        fig2 = px.bar(
            borough_avg,
            x="borough",
            y="avg_ridership",
            color="period",
            barmode="group",
            title="Average Ridership by Borough"
        )
        fig2.update_layout(
            width=950,
            height=380,
            xaxis_title="Borough",
            yaxis_title="Average Ridership",
            legend_title=""
        )
        display(fig2)

        pre_data = data[data["period"] == "Pre Policy"]
        post_data = data[data["period"] == "Post Policy"]

        pre_mean = pre_data["avg_ridership"].mean()
        post_mean = post_data["avg_ridership"].mean()

        if pd.notna(pre_mean) and pre_mean != 0:
            change_pct = (post_mean - pre_mean) / pre_mean * 100
            change_text = f"{change_pct:.2f}%"
        else:
            change_text = "N/A"

        total_riders = int(data["total_ridership"].sum())

        print(f"Average Ridership Change: {change_text}")
        print(f"Total Riders in Selection: {total_riders:,}")

In [9]:
peak_selector.observe(render_dashboard, names="value")
borough_selector.observe(render_dashboard, names="value")

controls = widgets.VBox([peak_selector, borough_selector])
layout = widgets.HBox([controls, output])

display(layout)
render_dashboard()